# Aula 08 — Sistema Multi-Agente com Memória Persistente

Este notebook evolui o sistema de triagem de e-mails da **Aula 07** adicionando
**memória persistente** via `langmem` e uma etapa de **Human-in-the-Loop** para
decisões de agendamento. O agente passa a:

1. **Salvar acompanhamentos** na memória ao responder e-mails
2. **Buscar contexto anterior** antes de redigir cada resposta
3. **Perguntar ao usuário** se deve agendar uma reunião (HITL)

## Diferenças em relação à Aula 07

| Aspecto | Aula 07 | Aula 08 |
|---------|---------|---------|
| Memória | Sem memória entre execuções | `InMemoryStore` com embeddings semânticos |
| Ferramentas de memória | — | `manage_memory_tool` + `search_memory_tool` |
| Roteamento | `Command(goto=...)` (LangGraph nativo) | `__next__` como chave customizada no estado |
| Prompts | Importados de módulo externo | Definidos inline no notebook |
| HITL | — | `human_in_the_loop_schedule` com verificação de memória |

## Arquitetura do Grafo

```
[email_input]
      |
      v
[triage_router] ---"ignore"/"notify"---> [END]
      |
   "respond"
      |
      v
[response_agent] <--------- [tool_node]
      |                          ^
      |--- tem tool_calls? -----/
      |          (write_email, schedule_meeting,
      |           check_calendar, manage_memory,
      |           search_memory)
      |
      v
 [InMemoryStore]  <-- armazena/busca por embeddings
      |
      v
[human_in_the_loop_schedule]
  (agendar reunião? sim/não)
```

> **`langmem`** é uma biblioteca que integra memória de longo prazo aos agentes LangGraph,
> abstraindo a lógica de criar, buscar e atualizar memórias no `Store`.

In [ ]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel
from typing_extensions import TypedDict, Literal, Annotated
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain.agents import AgentExecutor
from langgraph.graph import StateGraph, END, add_messages
from langgraph.prebuilt import ToolNode, create_react_agent  # create_react_agent: monta o sub-agente de resposta com loop ReAct interno
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings  # Embeddings: necessários para busca semântica no Store
from langmem import create_manage_memory_tool, create_search_memory_tool  # NOVO: biblioteca de memória persistente para agentes LangGraph
from langgraph.store.memory import InMemoryStore  # NOVO: store indexado por embeddings — armazena memórias na RAM
from IPython.display import Image, display
from langgraph.graph import StateGraph, END  # redeclarado abaixo (redundante, sem efeito colateral)
from langgraph.prebuilt import ToolNode

## 1. Imports

Além dos imports padrão do LangGraph e LangChain, este notebook introduz três
componentes novos de infraestrutura de memória:

| Import | Papel |
|--------|-------|
| `ChatGoogleGenerativeAI` | LLM principal para triagem e resposta |
| `GoogleGenerativeAIEmbeddings` | Gera embeddings para busca semântica nas memórias |
| `create_manage_memory_tool` | Ferramenta que salva/atualiza/deleta memórias no Store |
| `create_search_memory_tool` | Ferramenta que busca memórias por similaridade semântica |
| `InMemoryStore` | Store em memória do LangGraph indexado por embeddings |
| `create_react_agent` | Cria o sub-agente de resposta com loop ReAct interno |

> **Nota:** `StateGraph` e `ToolNode` aparecem importados duas vezes — a segunda
> importação é redundante, mas não causa problemas práticos.

In [ ]:
load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY')  # SDK do Google usa GOOGLE_API_KEY internamente
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

# Perfil do usuário — injetado no system prompt do agente para personalizar o comportamento
# Separado do código para facilitar adaptar o sistema para outros usuários sem alterar a lógica
profile = {
    "name": "Sarah",
    "full_name": "Sarah Chen",
    "user_profile_background": "Engenheira de software sênior liderando uma equipe de 5 desenvolvedores",
}

# Regras de triagem e instruções do agente — separadas do código como dicionário de configuração
# Permite ajustar o comportamento do agente sem modificar o grafo ou os nós
prompt_instructions = {
    "triage_rules": {
        "ignore": "Newsletters, spam, comunicados gerais da empresa",
        "notify": "Membro da equipe doente, notificações do sistema de build, atualizações de status de projeto",
        "respond": "Perguntas diretas de membros da equipe, solicitações de reunião, relatórios de bugs críticos",
    },
    "agent_instructions": """Você é um assistente executivo altamente eficiente. Execute as tarefas solicitadas de forma direta, sem diálogos desnecessários. Responda apenas com a chamada das ferramentas, a menos que seja estritamente necessário fornecer uma resposta textual.

Tarefas:
- Responder ao e-mail de entrada.
- Salvar a tarefa de acompanhamento na memória.
- Sugerir uma próxima ação, como 'Gostaria que eu agendasse uma reunião para discutir isso?'"""
}

## 2. Configuração do Ambiente e Perfil do Usuário

Carregamos as chaves de API do `.env` e definimos dois dicionários de configuração
que **parametrizam** o comportamento do agente sem alterar o código:

- **`profile`**: dados do usuário real (Sarah Chen) — injetados no system prompt do agente
- **`prompt_instructions`**: regras de triagem e instruções de comportamento do agente

Separar dados de configuração do código facilita adaptar o sistema para outros usuários
ou regras de negócio sem tocar na lógica do grafo.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# Router usa structured output: força o LLM a retornar JSON validado pelo Pydantic
# evitando a necessidade de parsing manual do texto livre do modelo
class Router(BaseModel):
    # Raciocínio por trás da classificação — útil para debug e auditoria da decisão
    reasoning: str
    # Enum de 3 valores: o LLM é obrigado a escolher exatamente um deles
    classification: Literal["ignore", "notify", "respond"]

# with_structured_output garante que o retorno do LLM respeite o schema Router
llm_router = llm.with_structured_output(Router)

# Prompt do sistema para triagem — usa placeholders {triage_no}, {triage_notify}, {triage_email}
# que serão preenchidos com as regras de prompt_instructions["triage_rules"] em tempo de execução
triage_system_prompt = """Você é um assistente de triagem de e-mails.
Classifique os e-mails como 'ignore', 'notify' ou 'respond'.

Regras:
- ignore: {triage_no}
- notify: {triage_notify}
- respond: {triage_email}
"""

# Prompt do usuário — injeta os dados do e-mail recebido para que o LLM possa classificá-lo
# O formato explícito De:/Para:/Assunto: ajuda o modelo a extrair os campos corretamente
triage_user_prompt = """
De: {author}
Para: {to}
Assunto: {subject}
Corpo da Mensagem:
{email_thread}
"""

## 3. LLM, Router e Prompts de Triagem

Definimos o LLM, o roteador estruturado e os templates de prompt para triagem.

**Diferença em relação à Aula 07:** os prompts são definidos diretamente aqui como
strings com placeholders `{...}`, enquanto na Aula 07 eram importados de um módulo
externo separado. A abordagem inline é mais adequada para notebooks de estudo.

### Fluxo do Roteador

```
e-mail recebido
       |
       v
  llm_router.invoke()
  (structured output: Router)
       |
   +---+---+---+
   |       |   |
 ignore  notify  respond
   |       |       |
  END     END  response_agent
```

`llm.with_structured_output(Router)` força o LLM a retornar um JSON validado pelo
Pydantic com os campos `reasoning` (raciocínio auditável) e `classification`
(enum de 3 valores). Isso elimina a necessidade de parsing manual do texto do LLM.

In [ ]:
# --- 3. Ferramentas ---
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Envia um e-mail para o destinatário especificado com o assunto e conteúdo fornecidos."""
    # Retorna JSON serializado — simula envio real; em produção chamaria uma API de e-mail
    return json.dumps({"to": to, "subject": subject, "content": content})

@tool
def schedule_meeting(attendees: list[str], subject: str, duration_minutes: int, preferred_day: str) -> str:
    """Agenda uma reunião com os participantes especificados."""
    return f"Reunião '{subject}' agendada para {preferred_day} com {len(attendees)} participantes"

@tool
def check_calendar_availability(day: str) -> str:
    """Verifica os horários disponíveis para o dia fornecido."""
    return f"Horários disponíveis em {day}: 9:00, 14:00, 16:00"

# Embeddings do Google para indexar as memórias no Store
# Permite busca semântica: encontra memórias relevantes mesmo com palavras diferentes da query
google_embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# InMemoryStore: armazena memórias em RAM indexadas por embeddings
# Em produção, seria substituído por um store persistente (Redis, Postgres, etc.)
store = InMemoryStore(index={"embed": google_embeddings})

# manage_memory_tool: permite ao agente criar, atualizar e deletar memórias no Store
# O namespace usa {langgraph_user_id} como placeholder dinâmico — resolvido via config
# Isso isola as memórias de cada usuário no mesmo Store compartilhado
manage_memory_tool = create_manage_memory_tool(
    store=store,
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)

# search_memory_tool: permite ao agente buscar memórias relevantes por similaridade semântica
# Usa os mesmos embeddings do Store para calcular a proximidade entre a query e as memórias armazenadas
search_memory_tool = create_search_memory_tool(
    store=store,
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)

# System prompt do agente de resposta — lista explicitamente as ferramentas disponíveis
# para que o LLM saiba quando e como chamá-las no ciclo ReAct
agent_system_prompt_memory = """
< Função >
Você é o(a) assistente executivo(a) de {full_name}. Sua prioridade é maximizar o desempenho de {name}.
{instructions}
</ Função >

< Ferramentas >
1. write_email(to, subject, content) - Envia e-mails
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Agenda reuniões
3. check_calendar_availability(day) - Verifica horários disponíveis
4. manage_memory - Armazena informações na memória
5. search_memory - Busca informações na memória
</ Ferramentas >
"""

# create_prompt é chamada a cada invocação do agente para injetar o perfil atual
# retorna [system_prompt] + histórico de mensagens do estado, construindo o contexto completo
def create_prompt(state):
    return [
        {"role": "system", "content": agent_system_prompt_memory.format(
            instructions=prompt_instructions["agent_instructions"], **profile
        )},
    ] + state['messages']

# Lista de todas as ferramentas disponíveis para o response_agent
# Inclui tanto ferramentas de negócio quanto as ferramentas de memória do langmem
tools = [write_email, schedule_meeting, check_calendar_availability, manage_memory_tool, search_memory_tool]

# create_react_agent cria um sub-grafo ReAct interno:
# llm → tool_calls? → tools → llm → ... → resposta final
# store=store conecta o agente ao InMemoryStore para que as ferramentas de memória funcionem
response_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=create_prompt,
    store=store
)

## 4. Ferramentas e Infraestrutura de Memória

Esta seção define as ferramentas do agente e configura o Store de memória semântica.

### Ferramentas de negócio

| Ferramenta | Descrição |
|------------|----------|
| `write_email` | Envia e-mail (simulado — retorna JSON com os dados) |
| `schedule_meeting` | Agenda reunião com lista de participantes |
| `check_calendar_availability` | Verifica horários livres em um dia |

### Infraestrutura de Memória

```
GoogleGenerativeAIEmbeddings (text-embedding-004)
              |
              v
        InMemoryStore
        index: {embed: embeddings}
              |
     namespace dinâmico:
     ("email_assistant", "{langgraph_user_id}", "collection")
              |
         +----+----+
         |         |
  manage_memory  search_memory
  (create/update/  (busca por
    delete)      similaridade semântica)
```

**`{langgraph_user_id}`** é um placeholder resolvido automaticamente pelo LangGraph
quando as ferramentas são invocadas com `config={"configurable": {"langgraph_user_id": "..."}}`.
Isso isola as memórias de cada usuário no mesmo Store compartilhado.

**`manage_memory_tool`** suporta as ações `create`, `update` e `delete` —
o agente decide qual usar com base no contexto da conversa.

**`search_memory_tool`** usa embeddings para busca semântica, não exata —
encontra memórias relevantes mesmo com wording diferente da query original.

**`create_prompt`** é chamada a cada invocação do agente para injetar dinamicamente
o perfil do usuário e as instruções no system prompt.

In [ ]:
# --- 4. Grafo de triagem ---
class State(TypedDict):
    email_input: dict                        # E-mail de entrada (from, to, subject, body)
    messages: Annotated[list, add_messages]  # Histórico acumulado — add_messages é o reducer

def triage_router(state: State) -> dict:
    # Extrai os campos do e-mail do estado para montar os prompts
    author = state['email_input']['from']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    body = state['email_input']['body']

    # Formata os prompts de triagem com as regras definidas em prompt_instructions
    system_prompt = triage_system_prompt.format(
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"]
    )
    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=body
    )
    # Invoca o roteador estruturado — retorna um objeto Router com reasoning + classification
    result = llm_router.invoke(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": user_prompt}]
    )

    if result.classification == "respond":
        # Serializa o e-mail como JSON para passá-lo como conteúdo da HumanMessage ao response_agent
        message_content = json.dumps({
            "sender": author,
            "recipient": to,
            "subject": subject,
            "body": body
        })
        # Retorna __next__ como chave no estado para sinalizar o próximo nó ao grafo
        # O LangGraph lê essa chave via lambda na add_conditional_edges (veja seção 6)
        return {
            "messages": [
                HumanMessage(content=f"Respond to the email {message_content}")
            ],
            "__next__": "response_agent"
        }
    elif result.classification in ["ignore", "notify"]:
        # Encerra o grafo sem ação adicional para e-mails de baixa prioridade
        return {"__next__": END}
    else:
        raise ValueError(f"Invalid classification: {result.classification}")

## 5. Estado do Grafo e Nó de Triagem

### Estado

| Campo | Tipo | Papel |
|-------|------|-------|
| `email_input` | `dict` | E-mail de entrada (from, to, subject, body) |
| `messages` | `Annotated[list, add_messages]` | Histórico da conversa com o agente |

> `add_messages` é um reducer especial do LangGraph: em vez de substituir a lista
> de mensagens, **acumula** novas mensagens ao histórico a cada nó executado.

### Roteamento via `__next__`

O nó `triage_router` retorna um dicionário com a chave `"__next__"` para indicar
o próximo nó a ser executado. Esta é uma **abordagem customizada** de roteamento:

```python
# O nó sinaliza o destino via chave no estado retornado:
return {"__next__": "response_agent"}  # ou END

# O grafo lê essa chave na add_conditional_edges:
lambda state: state.get("__next__", None)
```

**Diferença da Aula 07:** lá usava-se `Command(goto=...)` para roteamento imperativo
dentro do nó. Aqui, o roteamento é declarativo via chave no estado — mais simples
e mais fácil de inspecionar ao debugar.

In [ ]:
# --- 5. Construção do grafo ---
builder = StateGraph(State)
builder.add_node("triage_router", triage_router)
builder.add_node("response_agent", response_agent)
builder.add_node("tool_node", ToolNode(tools))  # Executa automaticamente as ferramentas da lista tools
builder.set_entry_point("triage_router")        # Ponto de entrada: todo e-mail passa pelo triage_router

# Roteamento a partir do triage_router: lê a chave __next__ do estado retornado pelo nó
# Diferença da Aula 07: aqui usamos lambda + __next__ em vez de Command(goto=...)
builder.add_conditional_edges(
    "triage_router",
    lambda state: state.get("__next__", None),  # Lê __next__ do estado
    {"response_agent": "response_agent", "END": END}
)

def route_to_tools(state):
    """Verifica se o agente fez chamadas de ferramenta e redireciona para o tool_node."""
    last_message = state["messages"][-1]
    # Se a última mensagem contém tool_calls, o ToolNode deve executá-las
    # Se não contém, o ciclo ReAct encerrou — vai para END
    if getattr(last_message, "tool_calls", None):
        return "tool_node"
    return END

# Loop ReAct do response_agent: agent → tools → agent → ... → END
builder.add_conditional_edges(
    "response_agent",
    route_to_tools,
    {"tool_node": "tool_node", END: END}
)
# tool_node sempre devolve ao response_agent após executar as ferramentas
builder.add_edge("tool_node", "response_agent")

email_agent = builder.compile()

## 6. Construção do Grafo

Montamos o `StateGraph` conectando os três nós com as arestas condicionais.

### Arestas do Grafo

```
triage_router
    |
    +-- __next__ == "response_agent" --> response_agent
    +-- __next__ == "END"            --> END

response_agent
    |
    +-- tem tool_calls? --> tool_node
    +-- sem tool_calls? --> END

tool_node --> response_agent  (loop até não haver mais tool_calls)
```

**`route_to_tools`** verifica se a última mensagem do agente contém `tool_calls`.
Se sim, redireciona para o `ToolNode` que executa as ferramentas e retorna
`ToolMessage`s com os resultados — fechando o ciclo ReAct do `response_agent`.

**`ToolNode(tools)`** é um nó pré-construído do LangGraph que executa automaticamente
qualquer ferramenta cujo nome apareça em `tool_calls` da última mensagem.

**Diferença da Aula 07:** lá usava-se `Command(goto=...)` dentro do nó para roteamento
imperativo. Aqui, o roteamento é declarativo via chave `__next__` no estado —
mais explícito e fácil de inspecionar ao debugar o grafo.

In [ ]:
# Gera a imagem do grafo em formato PNG
# xray=True expande os sub-grafos internos, revelando o loop ReAct do response_agent
display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

## 7. Visualizando o Grafo

O parâmetro `xray=True` expande os sub-grafos internos — em especial o sub-grafo
gerado pelo `create_react_agent` dentro do `response_agent` — revelando toda a
arquitetura com o loop ReAct interno (agent → tools → agent → ...).

In [ ]:
# --- 6. Human-in-the-Loop ---
def human_in_the_loop_schedule(email_sender, email_recipient, email_subject):
    # Verifica na memória se já existe uma reunião agendada para este remetente
    # Padrão: sempre checar memória ANTES de perguntar ao usuário — evita perguntas repetitivas
    search_results = search_memory_tool.invoke(
        {"query": f"Reunião agendada para {email_sender}"},
        config=config  # config passa o langgraph_user_id para resolver o namespace do Store
    )
    if isinstance(search_results, str):
        # search_memory_tool pode retornar string em caso de erro ou resultado vazio
        search_results = {}

    if search_results.get("results"):
        # Já existe reunião agendada — envia e-mail diretamente sem perguntar ao usuário
        email_content = write_email.invoke(
            {
                "to": email_sender,
                "subject": f"Re: {email_subject}",
                "content": "Olá, acabei de agendar uma conversa contigo para discutirmos esse assunto."
            },
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: write_email\nContent: {email_content}")
        return

    # Sem reunião na memória — pergunta ao usuário (ponto de intervenção humana)
    decision = input(f"Deseja agendar uma reunião para discutir o pedido de {email_sender}? (sim/não): ").strip().lower()
    
    if decision == "sim":
        # Agenda a reunião com os dois participantes
        meeting_output = schedule_meeting.invoke(
            {
                "attendees": [email_recipient.split('<')[0].strip(), email_sender.split('<')[0].strip()],
                "subject": f"Acompanhamento do pedido de {email_sender}",
                "duration_minutes": 30,
                "preferred_day": "amanhã"
            },
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: schedule_meeting\nContent: {meeting_output}")

        # Salva a reunião na memória para não perguntar novamente na próxima vez
        memory_output = manage_memory_tool.invoke(
            {"action": "create", "content": f"Reunião agendada para discutir o pedido de {email_sender}"},
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: manage_memory\nContent: {memory_output}")

        # Notifica o remetente por e-mail sobre a reunião agendada
        email_content = write_email.invoke(
            {
                "to": email_sender,
                "subject": f"Re: {email_subject}",
                "content": "Já agendei uma reunião contigo para discutirmos esse assunto."
            },
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: write_email\nContent: {email_content}")

    else:
        # Usuário optou por não agendar — envia e-mail de acompanhamento genérico
        email_content = write_email.invoke(
            {
                "to": email_sender,
                "subject": f"Re: {email_subject}",
                "content": "Estou acompanhando seu pedido e entrarei em contato assim que houver novidades."
            },
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: write_email\nContent: {email_content}")

        # Registra na memória que o acompanhamento foi feito — evita duplicar o e-mail
        memory_output = manage_memory_tool.invoke(
            {"action": "create", "content": f"E-mail de acompanhamento enviado para {email_sender}"},
            config=config
        )
        print("================== Tool Message ==================")
        print(f"Name: manage_memory\nContent: {memory_output}")

## 8. Função Human-in-the-Loop para Agendamento

Esta função implementa o padrão **HITL (Human-in-the-Loop)** para decisões de
agendamento, integrando memória para evitar perguntas redundantes ao usuário.

### Fluxo da função

```
Chamada com (email_sender, email_recipient, email_subject)
       |
       v
search_memory_tool: "Reunião agendada para {sender}?"
       |
   +---+---+
   |       |
 memória   sem memória
 existe         |
   |            v
   |    input("Deseja agendar reunião? sim/não")
   |            |
   |        +---+---+
   |        |       |
   |       sim      não
   |        |       |
   |   schedule_meeting    write_email (acompanhamento)
   |   + manage_memory     + manage_memory
   |   + write_email
   |
   +-> write_email (avisa que já há reunião agendada)
```

**Padrão de verificar memória antes de perguntar:** antes de acionar o `input()`,
a função consulta o Store para ver se já existe uma reunião agendada para esse
remetente. Isso evita perguntas repetitivas em fluxos encadeados e demonstra o
valor prático da memória persistente.

**`config=config`** é passado para as ferramentas de memória para que o
`{langgraph_user_id}` seja resolvido corretamente no namespace do Store.

In [ ]:
# --- 7. Exemplo de execução ---
# config define o user_id que resolve o placeholder {langgraph_user_id} no namespace do Store
# Todas as operações de memória desta sessão ficam isoladas sob o namespace "lance"
config = {"configurable": {"langgraph_user_id": "lance"}}

# Pré-popula a memória com contexto de uma interação anterior
# Simula um cenário real onde o agente já conhece o histórico sobre Alice Smith
# O response_agent poderá buscar este contexto com search_memory ao formular a resposta
initial_memory_content = "Acompanhamento necessário: Alice Smith perguntou sobre os endpoints de API ausentes na documentação do serviço de autenticação (/auth/refresh e /auth/validate). Sarah precisa revisar e esclarecer se eles foram intencionalmente omitidos ou se a documentação precisa de atualização."
manage_memory_tool.invoke({"action": "create", "content": initial_memory_content}, config=config)

email_input = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Acompanhamento",
    "body": "Olá Sarah, como está minha solicitação?"
}

# 1️⃣ Etapa 1: o grafo executa triage_router → response_agent → tool_node (em loop)
# O response_agent buscará a memória pré-populada para contextualizar a resposta à Alice
response = email_agent.invoke({"email_input": email_input}, config=config)
for msg in response["messages"]:
    if isinstance(msg, ToolMessage):  # Filtra apenas mensagens de resultado das ferramentas
        print("================== Mensagem da Ferramenta ==================")
        print(f"Nome: {msg.name}\nConteúdo: {msg.content}")

# 2️⃣ Etapa 2: Human-in-the-Loop — pergunta ao usuário se deve agendar uma reunião
# A função verifica primeiro a memória antes de exibir o input() para o usuário
human_in_the_loop_schedule(email_input["from"], email_input["to"], email_input["subject"])

## 9. Executando o Agente com Memória e HITL

Esta célula demonstra o fluxo completo em duas etapas:

### Pré-populate da memória

Antes de invocar o agente, inserimos manualmente uma memória com contexto anterior
sobre Alice Smith. Isso simula um cenário real onde o agente já tem histórico
de interações anteriores — permitindo ao `response_agent` buscar e usar esse
contexto ao formular a resposta.

### Etapa 1 — Agente responde ao e-mail

O `email_agent.invoke()` executa o grafo completo:
1. `triage_router` classifica o e-mail como `"respond"`
2. `response_agent` busca memórias relevantes com `search_memory`
3. `response_agent` redige o e-mail de resposta com `write_email`
4. `response_agent` salva um acompanhamento com `manage_memory`

### Etapa 2 — Human-in-the-Loop

`human_in_the_loop_schedule` verifica a memória e pergunta ao usuário se deve
agendar uma reunião. A resposta (`sim`/`não`) determina se `schedule_meeting`
é chamado e como o e-mail de retorno é formulado.

### Configuração `config`

```python
config = {"configurable": {"langgraph_user_id": "lance"}}
```

Este dicionário é passado para todas as chamadas do grafo e das ferramentas de
memória. O `langgraph_user_id` resolve o placeholder `{langgraph_user_id}` no
namespace do Store, garantindo que as memórias de `"lance"` fiquem isoladas das
memórias de outros usuários.